# Shift-Scan — Kaggle GPU — 20M-token HARPQ training (resumable)

This notebook launches **one 20-million-token HARPQ student run** using the existing `shift_scan_advanced_project.zip` source code.

**Runtime estimate:** at 920 tokens/sec, 20M tokens is about 6.0 hours of training, plus setup/evaluation. Actual throughput on your Kaggle GPU and session limits can vary; run the pilot below first.

This notebook is intended for **Kaggle Notebooks**:
- Select a GPU accelerator (T4 x2 / P100 / compatible GPU available to your account).
- Enable Internet in notebook settings to download the pretrained model and dataset.
- Attach a Kaggle Dataset containing `shift_scan_advanced_project.zip` as notebook input.
- Training outputs/checkpoints are written under `/kaggle/working/shift_scan_20m_runs`.

Kaggle working storage is session-scoped. Download the output notebook artifacts or use **Save Version** with outputs to preserve a checkpoint. For resume after a fresh session, attach the saved output folder as a Kaggle Dataset and set `RESUME_CHECKPOINT` in Section 3 to its `last.pt` path.

This is **one HARPQ run**, not a controlled comparison. Do not use it alone to claim HARPQ beats nearest rounding. Run matched nearest / HARPQ / FP-decay conditions with the same budget and seeds for such claims.

## 1. Kaggle GPU setup and source ZIP discovery

In Kaggle, add your source ZIP as a Dataset input. This cell searches `/kaggle/input` recursively for `shift_scan_advanced_project.zip`. Internet must be enabled for pretrained model/data downloads.

In [ ]:
import os, glob, zipfile, shutil, pathlib, json, time, subprocess, sys, textwrap

zip_candidates = glob.glob('/kaggle/input/**/shift_scan_advanced_project.zip', recursive=True)
assert zip_candidates, (
    'Could not find shift_scan_advanced_project.zip under /kaggle/input. '
    'Create a Kaggle Dataset containing this ZIP and attach it to this notebook.'
)
SOURCE_ZIP = zip_candidates[0]
print('Source ZIP:', SOURCE_ZIP)

WORKDIR = '/kaggle/working/shift_scan_20m'
os.makedirs(WORKDIR, exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as zf:
    zf.extractall(WORKDIR)

repo_candidates = glob.glob(os.path.join(WORKDIR, '**', 'train_qat.py'), recursive=True)
assert repo_candidates, 'train_qat.py was not found inside the source ZIP.'
REPO = os.path.dirname(repo_candidates[0])
os.chdir(REPO)
print('Using source:', REPO)
print('Files:', sorted(os.listdir(REPO))[:25])

## 2. Install dependencies

This uses the repository's requirements. If dependency installation changes core Kaggle packages unexpectedly, review the install output before proceeding.

In [ ]:
%pip -q install -r requirements.txt
%pip -q install -U 'transformers>=4.40' 'datasets>=2.18' 'huggingface_hub>=0.23'

## 3. Select the 20M run configuration

**Kaggle GPU starting profile:** sequence length 2048, micro-batch 2, gradient accumulation 8 (32,768 tokens per optimizer step). If CUDA runs out of memory, change `MICRO_BS=1` and `ACCUM=16` (same effective tokens/step, lower activation memory).

Use the same effective batch and settings across matched comparisons.

In [ ]:
# ---- Main experiment settings ----
RUN_NAME = 'harpq_h128_ternary_20m_seed0'
SEED = 0
TOKEN_BUDGET = 20_000_000  # target 20M; rounded down to complete optimizer steps below

# Kaggle GPU profile (lower memory fallback: MICRO_BS=1, ACCUM=16)
SEQ_LEN = 2048
MICRO_BS = 2
ACCUM = 8

# HARPQ + ternary student
DECAY = 'shift'
DECAY_QUANTIZER = 'horizon'
DECAY_HORIZON = 128
DECAY_LAG_WEIGHT = 'uniform'
TERNARY = 1
GROUP_SIZE = 128
ACTIVATION_AWARE = 1
STATE_WEIGHT = 0.05
STATE_LOSS_TYPE = 'hybrid'

DTYPE = 'fp16'
LEARNING_RATE = 1e-4
WARMUP_STEPS = 50
MIN_LR_FRAC = 0.1

# Kaggle session output location. Save Version / download outputs to preserve checkpoints.
OUT_ROOT = '/kaggle/working/shift_scan_20m_runs'
OUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

# Optional resume path: after a new session, attach a Kaggle Dataset containing prior output.
# Set to the exact attached last.pt path; leave None for fresh training.
RESUME_CHECKPOINT = None

TOKENS_PER_STEP = SEQ_LEN * MICRO_BS * ACCUM
TOTAL_STEPS = TOKEN_BUDGET // TOKENS_PER_STEP
ACTUAL_TOKENS = TOTAL_STEPS * TOKENS_PER_STEP
TOKEN_BUDGET = ACTUAL_TOKENS

print({
    'gpu_profile': 'Kaggle GPU',
    'gpu': __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'NO CUDA',
    'tokens_per_step': TOKENS_PER_STEP,
    'optimizer_steps': TOTAL_STEPS,
    'actual_token_budget': ACTUAL_TOKENS,
    'approx_budget_M': round(ACTUAL_TOKENS / 1e6, 3),
    'output_dir': OUT_DIR,
    'resume_checkpoint': RESUME_CHECKPOINT
})

## 4. GPU and source checks

Do not start the training run unless CUDA is available and repository tests pass.

In [ ]:
import torch, subprocess, os, json, platform
assert torch.cuda.is_available(), 'CUDA GPU not found. In Kaggle, select a GPU accelerator in Notebook Settings.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('Python:', platform.python_version())

# Fast repository correctness check before the expensive training run.
test = subprocess.run([sys.executable, 'tests.py'], text=True)
if test.returncode != 0:
    raise RuntimeError('Repository tests failed; do not launch the 100M run.')

## 5. Estimate runtime from a short throughput pilot

Run a short 20-step pilot first. It measures throughput on the Kaggle GPU available to your session. Use the measured estimate to decide whether the complete 20M-token budget fits your available session time.

In [ ]:
# Optional: run this short pilot before the full 100M experiment.
# Set RUN_PILOT=False only if you have already verified this exact configuration.
RUN_PILOT = True
PILOT_STEPS = 20
if RUN_PILOT:
    pilot_tokens = PILOT_STEPS * TOKENS_PER_STEP
    pilot_dir = os.path.join(OUT_ROOT, RUN_NAME + '_pilot')
    os.makedirs(pilot_dir, exist_ok=True)
    pilot_cmd = [
        sys.executable, 'train_qat.py',
        '--name', RUN_NAME + '_pilot', '--out', OUT_ROOT,
        '--pretrained', 'state-spaces/mamba2-130m',
        '--tokens', str(pilot_tokens), '--seq-len', str(SEQ_LEN),
        '--micro-bs', str(MICRO_BS), '--accum', str(ACCUM),
        '--seed', str(SEED), '--dtype', DTYPE,
        '--lr', str(LEARNING_RATE), '--warmup', str(min(WARMUP_STEPS, PILOT_STEPS)),
        '--decay', DECAY, '--decay-quantizer', DECAY_QUANTIZER,
        '--decay-horizon', str(DECAY_HORIZON), '--decay-lag-weight', DECAY_LAG_WEIGHT,
        '--ternary', str(TERNARY), '--group-size', str(GROUP_SIZE),
        '--activation-aware', str(ACTIVATION_AWARE),
        '--state-weight', str(STATE_WEIGHT), '--state-loss-type', STATE_LOSS_TYPE,
        '--log-every', '20', '--eval-every', str(PILOT_STEPS),
        '--save-every', str(PILOT_STEPS)
    ]
    print('Pilot command:', ' '.join(pilot_cmd))
    t0 = time.time()
    subprocess.run(pilot_cmd, check=True)
    elapsed = time.time() - t0
    measured_tps = pilot_tokens / max(elapsed, 1)
    print(f'Pilot throughput: {measured_tps:,.0f} tokens/s')
    print(f'Rough 20M training-time estimate: {TOKEN_BUDGET / measured_tps / 3600:.1f} hours')
else:
    print('Pilot skipped by user choice.')

## 6. Launch or resume the 20M-token run

The trainer writes `last.pt` every 100 optimizer steps. Kaggle sessions may stop at their session limit; save the notebook version/output or download the run folder. For a new session, attach the prior output as a Kaggle Dataset and set `RESUME_CHECKPOINT` in Section 3 to that `last.pt`.

Do not change architecture, quantization, sequence length, batch settings, seed, or dtype when resuming.

In [ ]:
CHECKPOINT = os.path.join(OUT_DIR, 'last.pt')
cmd = [
    sys.executable, 'train_qat.py',
    '--name', RUN_NAME, '--out', OUT_ROOT,
    '--pretrained', 'state-spaces/mamba2-130m',
    '--tokens', str(TOKEN_BUDGET), '--seq-len', str(SEQ_LEN),
    '--micro-bs', str(MICRO_BS), '--accum', str(ACCUM),
    '--seed', str(SEED), '--dtype', DTYPE,
    '--lr', str(LEARNING_RATE), '--warmup', str(WARMUP_STEPS),
    '--min-lr-frac', str(MIN_LR_FRAC),
    '--decay', DECAY, '--decay-quantizer', DECAY_QUANTIZER,
    '--decay-horizon', str(DECAY_HORIZON), '--decay-lag-weight', DECAY_LAG_WEIGHT,
    '--ternary', str(TERNARY), '--group-size', str(GROUP_SIZE),
    '--activation-aware', str(ACTIVATION_AWARE),
    '--state-weight', str(STATE_WEIGHT), '--state-loss-type', STATE_LOSS_TYPE,
    '--log-every', '20', '--eval-every', '500', '--save-every', '100',
    '--val-seqs', '32'
]
if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    cmd += ['--resume', RESUME_CHECKPOINT]
    print('Attached checkpoint found: RESUMING')
elif os.path.exists(CHECKPOINT):
    cmd += ['--resume', CHECKPOINT]
    print('Existing checkpoint found: RESUMING')
else:
    print('No checkpoint found: starting from step 0')

print('Launching 20M-token HARPQ run...')
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('Training process completed.')

## 7. Inspect training logs and verify the final checkpoint

In [ ]:
log_path = os.path.join(OUT_DIR, 'log.jsonl')
if os.path.exists(log_path):
    rows = [json.loads(x) for x in open(log_path) if x.strip()]
    print('Log records:', len(rows))
    for row in rows[-10:]:
        print(row)
else:
    print('No log yet:', log_path)

print('Checkpoint exists:', os.path.exists(CHECKPOINT))
if os.path.exists(CHECKPOINT):
    ck = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
    extra = ck.get('extra', {})
    print('Saved step:', extra.get('step'))
    print('Saved tokens:', extra.get('tokens'))

## 8. What this run does — and does not — establish

- It provides a controlled-scale 20M-token HARPQ run for initial quantization/decay evaluation.
- It does **not** prove HARPQ is better than nearest rounding.
- For defensible comparisons, run matched baseline with `--decay-quantizer nearest` and floating-point decay ablation (`--decay exp`), keeping the same dataset split, token budget, sequence length, effective batch, seed, dtype and evaluation.
- Do not claim speedup from ternary weights without real hardware benchmarks; the current Triton ternary GEMV may not be faster than FP16.
- Keep `args.json`, `run_manifest.json`, `log.jsonl`, `last.pt`, and evaluation outputs together. On Kaggle, download or Save Version outputs to preserve them.